# 29 · Fatores da via dentro das malhas: o que serve de parâmetro do EAR

> **Regra da disciplina (aula 3):** especificação FOCUS → código gerado pelo assistente → executar → *gate* contra uma resposta já conhecida → aceitar.
> **Quando o gate falha, a restrição que faltou entra em `U`. O código não é remendado.**
>
> Cada passo abaixo tem três partes: a especificação FOCUS enviada ao assistente, o gate declarado **antes** do código (com a fonte da resposta conhecida e da tolerância) e o código que voltou, executado e gateado. O registro de uso de IA é exportado ao final (`registro_uso_ia.csv`).

**Por que este passo.** A Declaração de Rede traz, por trecho, bitola, velocidade, rampa, carga por eixo, tráfego vinculado, dormente, trilho, fixação, taxa de dormentação, sinalização, produto perigoso e número de linhas; um EAR de ferrovia nova conhece esses valores no projeto. Este notebook separa o que muda o risco dentro de uma ferrovia (candidato a parâmetro do EAR) do que só distingue uma ferrovia da outra, com classes e regra de decisão declaradas antes, risco relativo de Mantel-Haenszel estratificado por malha, faixa de pátio e classe de raio, ganho em trechos inteiros fora do ajuste (30 sorteios) e placebo.

**Gates, declarados antes do código:** 20.936 células; 2.123 km atingidos; regra FICO 0,588; classe de referência com risco relativo 1; classes somam 20.936; placebo com ganho de no máximo 0,01. O placebo falhou na primeira execução (0,016) e a especificação recebeu o efeito fixo da malha (registrado no código).

In [ ]:
# === Setup: caminhos, bibliotecas e as três ferramentas (gate, gate_between, log_ia) ===
import os, sys, json, re, unicodedata, datetime as dt
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if os.environ.get("ET261400_BASE"):
    BASE = os.environ["ET261400_BASE"]                # definido pelo notebook 00 (execução em lote) ou pela execução local
elif IN_COLAB and os.path.exists("/content/ET261400/data"):
    BASE = "/content/ET261400"                        # pacote já descompactado na sessão do Colab
elif IN_COLAB and os.path.exists("/content/et261400_colab.zip"):
    import zipfile
    zipfile.ZipFile("/content/et261400_colab.zip").extractall("/content/ET261400")   # pacote enviado pelo painel de arquivos
    BASE = "/content/ET261400"
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE = "/content/drive/MyDrive/ET261400"          # pasta do projeto no Drive (ajustar se for outra)
else:
    BASE = os.environ.get("ET261400_BASE", os.path.abspath(".."))   # execução local: raiz do repositório
RAW, PROC = f"{BASE}/data/raw", f"{BASE}/data/processed"
FIG, TAB = f"{BASE}/results/figures", f"{BASE}/results/tables"
for d in (PROC, FIG, TAB):
    os.makedirs(d, exist_ok=True)
print("BASE =", BASE)

_GATES = []          # todo gate executado fica registrado aqui
_LOG_IA = []         # registro de uso de IA: passo, ferramenta, especificação, resultado

def gate(name, got, expected, tol=0.0, source=""):
    """Compara um número calculado com um número que já se tinha antes do código.
    tol vem da fonte (meio passo de arredondamento, faixa publicada), nunca é alargada até passar."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = abs(got - expected) <= tol
    _GATES.append(dict(gate=name, got=got, expected=expected, tol=tol, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, esperado {expected} (tol +/-{tol}) {('| ' + source) if source else ''}")
    return ok

def gate_between(name, got, low, high, source=""):
    """O mesmo, quando a fonte publica uma faixa e não um número."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = low <= got <= high
    _GATES.append(dict(gate=name, got=got, expected=f"[{low}, {high}]", tol=None, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, faixa [{low}, {high}] {('| ' + source) if source else ''}")
    return ok

def log_ia(passo, ferramenta, especificacao, resultado, verificacao):
    """Registro de uso de IA exigido pela disciplina: o que foi pedido, a que ferramenta, o que voltou e como foi verificado."""
    _LOG_IA.append(dict(data=dt.date.today().isoformat(), passo=passo, ferramenta=ferramenta,
                        especificacao=especificacao.strip(), resultado=resultado, verificacao=verificacao))

def exportar_registros(prefixo):
    pd.DataFrame(_GATES).to_csv(f"{TAB}/{prefixo}_gates.csv", index=False)
    pd.DataFrame(_LOG_IA).to_csv(f"{TAB}/{prefixo}_registro_uso_ia.csv", index=False)
    n_ok = sum(g["ok"] for g in _GATES)
    print(f"gates: {n_ok}/{len(_GATES)} PASS | registros de IA: {len(_LOG_IA)} | exportados em {TAB}")

def norm_txt(s):
    """Maiúsculas sem acento, espaços simples: chave de junção para nomes de pátios e estações."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).upper().strip()


In [ ]:
import warnings; warnings.filterwarnings("ignore")
from scipy import sparse
from scipy.stats import rankdata
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import matplotlib; matplotlib.rcParams.update({"font.family": "DejaVu Sans", "axes.spines.top": False, "axes.spines.right": False})
import matplotlib.pyplot as plt
SEED = 42
C0 = pd.read_csv(f"{PROC}/11_celulas_1km.csv", low_memory=False)
C = C0[~C0["sentido"].astype(str).str.contains("Interromp")].sort_values(["ferrovia", "linha", "km_a"]).reset_index(drop=True); gate("células em operação", len(C), 20936, source="notebook 11")
E = pd.read_csv(f"{PROC}/11_eventos_celulas.csv"); E = E[E["cel_id"].isin(C["cel_id"])].copy(); E["ano"] = E["ano"].astype(int); EV = E[E["via"]]; gate("descarrilamentos por via", len(EV), 2681, source="notebook 11")
N = len(C); idx = pd.Series(np.arange(N), index=C["cel_id"].values); ANOS = np.arange(2011, 2026)
def matriz(ev):
    M_ = np.zeros((15, N))
    for k in range(15):
        s = ev[ev["ano"] == 2011 + k].groupby("cel_id").size(); M_[k, idx[s.index].values] = s.values
    return M_
CV_, CA_ = matriz(EV), matriz(E)
def conta(M_, a0, a1): return M_[(ANOS >= a0) & (ANOS <= a1)].sum(0)
EARN = ["dmin_patio", "dmin_amv", "frac_zona_patio"] + [f"f16p{j}" for j in range(5)] + [f"f16a{j}" for j in range(5)] + ["rmin_geo"] + [f"fr_lt{l}" for l in (300, 400, 500, 600, 800)]
NUM = EARN + ["rampa_max_pct", "carga_eixo_t", "taxa_dorm", "gab_vert_m", "gab_horiz_m", "cap_vinc", "dias_op", "raio_min_m", "vma_carregado"]
CAT = ["dormente", "perfil_trilho", "fixacao", "sinalizacao", "controle_circ", "bitola", "sentido", "n_linhas"]
for c in NUM: C[c] = pd.to_numeric(C[c], errors="coerce")
for c in CAT: C[c] = C[c].astype(str).str.strip().replace({"nan": "NA", "": "NA"})
def design(num, cat):
    X = C[num].copy(); med = X.median().fillna(0.0)
    for c in num: X[c + "_na"] = X[c].isna().astype(float); X[c] = X[c].fillna(med[c])
    X = pd.concat([X, pd.get_dummies(C[cat], dtype=float)], axis=1) if cat else X; return StandardScaler().fit_transform(X.values.astype(float))
X0, XE = design(NUM, CAT), design(EARN, [])
FERC = pd.factorize(C["ferrovia"])[0]; LINC = pd.factorize(C["ferrovia"].astype(str) + "|" + C["linha"].astype(str))[0]; SEGC = pd.factorize(C["seg_id"].astype(str))[0]; KM = ((C["km_a"] + C["km_b"]) / 2).values; LN = C["len_km"].values
ri, ci = [], []
for l in range(LINC.max() + 1):
    i = np.where(LINC == l)[0]; k = KM[i]; M_ = (np.abs(k[:, None] - k[None, :]) <= 20) & ~np.eye(len(i), dtype=bool); a, b = np.where(M_); ri.append(i[a]); ci.append(i[b])
W = sparse.csr_matrix((np.ones(sum(len(a) for a in ri)), (np.concatenate(ri), np.concatenate(ci))), shape=(N, N)); nviz = np.asarray(W.sum(1)).ravel()
def historico(a0, a1):
    n = a1 - a0 + 1; hv = conta(CV_, a0, a1); ha = conta(CA_, a0, a1); b = (hv > 0).astype(float); d = pd.DataFrame({"b": b, "hv": hv})
    return np.column_stack([hv / n, ha / n, b, (W @ b) / np.maximum(nviz, 1), (W @ hv) / np.maximum(nviz, 1) / n, d.groupby(SEGC)["b"].transform("mean").values, d.groupby(LINC)["b"].transform("mean").values, d.groupby(FERC)["b"].transform("mean").values, d.groupby(FERC)["hv"].transform("mean").values / n])
dp = C["dmin_patio"].fillna(1e9).values; rg = C["rmin_geo"].fillna(1e6).values
B2 = 0.5 * np.select([dp <= 300, dp <= 600, dp <= 1200, dp <= 1800], [4, 3, 2, 1], 0) / 4 + 0.5 * np.select([rg < 500, rg < 1200, rg < 2300], [3, 2, 1], 0) / 3
def auc(y, s):
    n1 = y.sum(); n0 = len(y) - n1; return float((rankdata(s)[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
def auc_dentro(s, y):
    a, w = [], []
    for f in range(FERC.max() + 1):
        i = FERC == f
        if 5 <= y[i].sum() < i.sum(): a.append(auc(y[i], s[i])); w.append(y[i].sum())
    return float(np.average(a, weights=w))
def lr(Xtr, ytr, Xte): return LogisticRegression(C=0.5, max_iter=3000).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
DES = np.random.default_rng(SEED).random(N)
DES = np.random.default_rng(SEED).random(N)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import average_precision_score
log_ia("29.0 dados e funções", "Claude", "FOCUS: mesmas funções, matriz e protocolo dos notebooks 24 e 25", "design, auc, lr", "gates de contagem")

# 0. Resposta e protocolo: iguais ao notebook 23 e à dissertação (trechos inteiros entre pátios fora do ajuste, 5 grupos, 30 sorteios)
n15 = conta(CV_, 2011, 2025); y = (n15 > 0).astype(int); gate("quilômetros com descarrilamento por via em 2011 a 2025", int(y.sum()), 2123, source="notebook 11")
gate("regra do EAR no período inteiro", round(auc(y, B2), 3), 0.588, tol=0.002, source="notebook 11 e dissertação v4")
KF, R = 5, 30
def folds_por(grupo, seed, k=KF):
    u = np.unique(grupo); f = dict(zip(u, np.random.default_rng(seed).permutation(len(u)) % k)); return np.array([f[g] for g in grupo])
def auc_grupo(s, yy, G, minimo=5):
    a, w = [], []
    for g in np.unique(G):
        i = G == g
        if minimo <= yy[i].sum() < i.sum(): a.append(auc(yy[i], s[i])); w.append(yy[i].sum())
    return float(np.average(a, weights=w))
def topo(s, yy):
    o = np.lexsort((DES, -s)); top = np.zeros(N, bool); top[o[(np.cumsum(LN[o]) / LN.sum()) <= 0.20]] = True; return top
def matriz_conf(s, yy):
    t = topo(s, yy); tp = int((t & (yy == 1)).sum()); fp = int((t & (yy == 0)).sum()); fn = int((~t & (yy == 1)).sum()); tn = int((~t & (yy == 0)).sum())
    assert tp + fn == yy.sum() and tp + fp + fn + tn == N
    rec = tp / (tp + fn); pre = tp / (tp + fp); return dict(VP=tp, FP=fp, FN=fn, VN=tn, recall=rec, precisao=pre, especificidade=tn / (tn + fp), F1=2 * pre * rec / (pre + rec))




In [ ]:
# 29 · Fatores da via dentro das malhas: quais servem de parâmetro do EAR e quais só separam ferrovias
# FOCUS (declarado antes do código, 22/11/2026, 23h20)
#  F: a Declaração de Rede traz, por trecho, bitola, velocidade, rampa, carga por eixo, capacidade vinculada (tráfego), dormente, trilho, fixação,
#     taxa de dormentação, sinalização, produto perigoso e número de linhas. Um EAR de ferrovia nova conhece esses valores no projeto.
#  O: separar, para cada fator, o que muda o risco de descarrilamento por via DENTRO de uma ferrovia (candidato a parâmetro do EAR)
#     do que só distingue uma ferrovia da outra (operador, manutenção, registro), que não se transfere a uma linha nova.
#  C: matriz de células e eventos do notebook 11; faixas aferidas no notebook 27 (pátio 250, 1.000, 1.500 e 3.000 m; raio 250, 300 e 600 m).
#  U: só dados públicos; classes de cada fator e regra de decisão escritas antes de rodar; mesmo protocolo dos notebooks 24 e 27
#     (trechos inteiros entre pátios fora do ajuste, 5 grupos, 30 sorteios, sementes 42 a 71); nada é ajustado para atingir alvo.
#  Regra de decisão (antes dos resultados): o fator entra no EAR v3 se
#    (a) for estimável dentro das malhas: em pelo menos 3 malhas há 2 classes com 5% ou mais das células (sem contar "sem dado");
#    (b) alguma classe tiver risco relativo ajustado com IC 95% que exclui 1 (Mantel-Haenszel estratificado por malha, faixa de pátio e classe de raio aferidas);
#    (c) somado à regra aferida, elevar a AUC dentro das malhas em trechos fora do ajuste, com IC 95% pareado acima de zero.
#  Restrição acrescentada pelo gate loop (depois da primeira execução, em que G6 falhou: 0,016): no item 3, a base e a base + fator incluem o efeito fixo
#    da malha (indicadoras). Sem ele, qualquer variável que marque a malha, mesmo embaralhada, melhora a ordenação dentro das malhas porque corrige os pesos
#    das faixas, confundidos com as diferenças entre malhas; o ganho medido seria desse ajuste, não do fator. A tolerância do G6 não mudou.
#  Gates: G1 20.936 células; G2 2.123 km atingidos; G3 regra FICO 0,588; G4 RR da classe de referência = 1; G5 células por classe somam 20.936;
#         G6 placebo: o fator embaralhado dentro de cada malha muda a AUC dentro das malhas em no máximo 0,01 (média dos sorteios).
import time
t0 = time.time()
def faixas_p(d, c): return np.select([d <= c[0], d <= c[1], d <= c[2], d <= c[3]], [4, 3, 2, 1], 0)
def classes_r(r, c): return np.select([r < c[0], r < c[1], r < c[2]], [3, 2, 1], 0)
FICO_P, FICO_R, AFER_P, AFER_R, W_AFER = (300, 600, 1200, 1800), (500, 1200, 2300), (250, 1000, 1500, 3000), (250, 300, 600), 0.6
gate("G1 células em operação", N, 20936, source="notebook 11")
gate("G2 km com descarrilamento por via, 2011 a 2025", int(y.sum()), 2123, source="notebook 11")
gate("G3 regra do EAR (FICO) reconstruída", round(auc(y, 0.5 * faixas_p(dp, FICO_P) / 4 + 0.5 * classes_r(rg, FICO_R) / 3), 3), 0.588, tol=0.002, source="notebooks 24 e 27")
KPA, KRA = faixas_p(dp, AFER_P), classes_r(rg, AFER_R)                 # faixa de pátio (0 a 4) e classe de raio (0 a 3) aferidas
S_AFER = W_AFER * KPA / 4 + (1 - W_AFER) * KRA / 3

# 1. Classes de cada fator, declaradas antes (cortes de projeto; zero na Declaração de Rede = sem dado, exceto rampa)
def cls_num(col, cortes, rot, zero_na=True):
    v = pd.to_numeric(C[col], errors="coerce").values.astype(float)
    if zero_na: v[v <= 0] = np.nan
    k = np.digitize(np.nan_to_num(v, nan=-1), cortes, right=True); out = np.array(rot, dtype=object)[np.minimum(k, len(rot) - 1)]; out[np.isnan(v)] = "sem dado"; return out.astype(str)
def txt(col, mapa=None):                                                         # texto com ausente explícito (o pandas recente mantém NaN em colunas de texto)
    s = C[col].astype(object).where(C[col].notna(), "sem dado").astype(str).str.strip().replace({"NA": "sem dado", "nan": "sem dado", "": "sem dado"})
    return (s.replace(mapa) if mapa else s).values
PERF = C["perfil_trilho"].astype(str).str.extract(r"(\d+)")[0].astype(float).values
SIN = pd.Series(txt("sinalizacao"))
FAT = {
    "Bitola": txt("bitola"),
    "Velocidade máxima carregado (km/h)": cls_num("vma_carregado", [30, 40, 60], ["até 30", "31 a 40", "41 a 60", "acima de 60"]),
    "Rampa máxima (%)": cls_num("rampa_max_pct", [1.0, 1.5, 2.0], ["até 1,0", "1,0 a 1,5", "1,5 a 2,0", "acima de 2,0"], zero_na=False),
    "Carga por eixo (t)": cls_num("carga_eixo_t", [20, 25, 30], ["até 20", "20 a 25", "25 a 30", "acima de 30"]),
    "Tráfego (trens por dia vinculados)": np.where(pd.to_numeric(C["cap_vinc"], errors="coerce").fillna(0).values <= 0, "zero vinculado",
                                                   cls_num("cap_vinc", [1, 3, 10], ["até 1", "1 a 3", "3 a 10", "acima de 10"])),
    "Dormente": txt("dormente", {"Mad/Conc": "misto", "Mad/Aço": "misto"}),
    "Trilho": np.where(np.isnan(PERF), "sem dado", np.select([PERF <= 45, PERF <= 57], ["leve (até TR 45)", "médio (TR 50 a 57)"], "pesado (TR 60 ou mais)")),
    "Fixação": txt("fixacao"),
    "Taxa de dormentação (por km)": cls_num("taxa_dorm", [1667, 1750, 1790], ["até 1.667", "1.668 a 1.750", "1.751 a 1.790", "acima de 1.790"]),
    "Sinalização": np.select([SIN.str.contains("não sinal", case=False), SIN.eq("ABS"), SIN.isin(["CTC", "CBTC", "ATC"])], ["não sinalizado", "ABS", "CTC, CBTC ou ATC"], "TWE ou outro"),
    "Produto perigoso": txt("produto_perigoso"),
    "Número de linhas": txt("n_linhas"),
}
for k_, v_ in FAT.items(): gate(f"G5 {k_}: células por classe somam a malha", int(len(v_)), N, source="definição")
log_ia("29.1 classes", "Claude", "FOCUS: classes de cada fator por cortes de projeto declarados antes", "dicionário FAT", "somas")

# 2. Estimabilidade, risco relativo ajustado e taxa nacional por classe
NL = LINC.max() + 1; ESTR = pd.factorize(pd.Series(FERC).astype(str) + "|" + pd.Series(KPA).astype(str) + "|" + pd.Series(KRA).astype(str))[0]
def rr_mh_s(cls, K, ref, yy, w, S):
    ns = S.max() + 1; j = S * K + cls
    n_ = np.bincount(j, weights=w, minlength=ns * K).reshape(ns, K); a_ = np.bincount(j, weights=w * yy, minlength=ns * K).reshape(ns, K); out = np.full(K, np.nan)
    for k in range(K):
        T_ = n_[:, k] + n_[:, ref]; ok = T_ > 0; num = (a_[ok, k] * n_[ok, ref] / T_[ok]).sum(); den = (a_[ok, ref] * n_[ok, k] / T_[ok]).sum(); out[k] = num / den if den > 0 else np.nan
    return out
LIN_RR, LIN_EST = [], []
rng_ = np.random.default_rng(SEED); WB = [np.bincount(rng_.choice(NL, NL), minlength=NL)[LINC].astype(float) for _ in range(300)]
for nome, v in FAT.items():
    rot = [c for c in pd.Series(v).value_counts().index if c != "sem dado"] + (["sem dado"] if (v == "sem dado").any() else [])
    cod = np.array([rot.index(x) for x in v]); K = len(rot); ref = 0                      # referência: a classe com mais células
    t = pd.crosstab(FERC, v); t = t.drop(columns=[c for c in t.columns if c == "sem dado"]); frac = t.div(t.sum(1), axis=0)
    n_malhas = int(((frac >= 0.05).sum(1) >= 2).sum()); entre = 1 - float((frac.max(1) * t.sum(1)).sum() / t.values.sum())
    rr = rr_mh_s(cod, K, ref, y, np.ones(N), ESTR); rr_m = rr_mh_s(cod, K, ref, y, np.ones(N), FERC)
    B_ = np.array([rr_mh_s(cod, K, ref, y, w_, ESTR) for w_ in WB])
    for k in range(K):
        i = cod == k
        LIN_RR.append(dict(fator=nome, classe=rot[k], referencia=rot[ref], celulas=int(i.sum()), km_atingidos=int(y[i].sum()), taxa_nacional=float(y[i].mean()),
                           malhas_com_a_classe=int((t[rot[k]] > 0).sum()) if rot[k] in t.columns else 0,
                           rr_bruto_nacional=float(y[i].mean() / y[cod == ref].mean()), rr_dentro_malhas=float(rr_m[k]), rr_ajustado=float(rr[k]),
                           ic_inf=float(np.nanpercentile(B_[:, k], 2.5)), ic_sup=float(np.nanpercentile(B_[:, k], 97.5))))
    LIN_EST.append(dict(fator=nome, classes=K, malhas_com_2_classes=n_malhas, celulas_fora_da_classe_dominante_da_malha=entre, estimavel=n_malhas >= 3))
RRF = pd.DataFrame(LIN_RR); EST = pd.DataFrame(LIN_EST)
RRF["ic_exclui_1"] = (RRF["classe"] != RRF["referencia"]) & (RRF["classe"] != "sem dado") & ((RRF["ic_inf"] > 1) | (RRF["ic_sup"] < 1))
gate("G4 RR da classe de referência", float(RRF.loc[RRF["classe"] == RRF["referencia"], "rr_ajustado"].abs().max()), 1.0, tol=1e-9, source="definição")
display(EST.round(3)); display(RRF.round(3)); RRF.to_csv(f"{TAB}/29_risco_relativo_fatores.csv", index=False)
print("rr ok", round(time.time() - t0), "s"); sys.stdout.flush()
log_ia("29.2 risco relativo", "Claude", "FOCUS: Mantel-Haenszel por malha, faixa de pátio e classe de raio; IC por 300 reamostragens de linhas", "29_risco_relativo_fatores.csv", "referência = 1")

# 3. Ganho dentro das malhas ao somar cada fator à regra aferida (trechos inteiros fora; 30 sorteios) e placebo
def oh(v): return pd.get_dummies(pd.Series(v).astype(str), dtype=float).iloc[:, 1:].values
XB0 = np.column_stack([oh(KPA), oh(KRA)])                                       # sem malha: usado só na malha inteira fora (item 4), em que a malha é desconhecida
XB = np.column_stack([XB0, oh(FERC)])                                            # com efeito fixo da malha: base do ganho dentro das malhas (item 3)
XF = {k_: oh(v_) for k_, v_ in FAT.items()}
def placebo(v, seed):                                                           # embaralha o fator dentro de cada malha (mantém a distribuição por malha)
    r_ = np.random.default_rng(seed); v2 = np.array(v, dtype=object).copy()
    for f in range(FERC.max() + 1):
        i = np.where(FERC == f)[0]; v2[i] = v2[r_.permutation(i)]
    return oh(v2.astype(str))
LIN_G, GU29 = [], {}
for r in range(R):
    fo = folds_por(SEGC, SEED + r); S = {"base": np.zeros(N)}; S.update({k_: np.zeros(N) for k_ in FAT}); S["placebo"] = {k_: np.zeros(N) for k_ in FAT} if r < 10 else None
    XP = {k_: placebo(v_, 1000 + r) for k_, v_ in FAT.items()} if r < 10 else None
    for k in range(KF):
        te = fo == k; tr = ~te
        S["base"][te] = lr(XB[tr], y[tr], XB[te])
        for k_ in FAT:
            Xk = np.column_stack([XB, XF[k_]]); S[k_][te] = lr(Xk[tr], y[tr], Xk[te])
            if r < 10: Xp = np.column_stack([XB, XP[k_]]); S["placebo"][k_][te] = lr(Xp[tr], y[tr], Xp[te])
    ab, am = auc(y, S["base"]), auc_grupo(S["base"], y, FERC)
    for k_ in FAT:
        LIN_G.append(dict(sorteio=r + 1, fator=k_, auc_pais_base=ab, auc_malhas_base=am, auc_pais=auc(y, S[k_]), auc_malhas=auc_grupo(S[k_], y, FERC),
                          placebo_malhas=auc_grupo(S["placebo"][k_], y, FERC) if r < 10 else np.nan))
    if r < 10: GU29[r] = {k_: S[k_] for k_ in ["base"] + list(FAT)}
    if r in (0, 9, 29): print("sorteio", r + 1, round(time.time() - t0), "s"); sys.stdout.flush()
G29 = pd.DataFrame(LIN_G); G29["ganho_malhas"] = G29["auc_malhas"] - G29["auc_malhas_base"]; G29["ganho_pais"] = G29["auc_pais"] - G29["auc_pais_base"]; G29["placebo_ganho"] = G29["placebo_malhas"] - G29["auc_malhas_base"]
G29.to_csv(f"{TAB}/29_ganho_por_sorteio.csv", index=False)
gate_between("G6 placebo: maior ganho médio com o fator embaralhado dentro da malha", round(float(G29.groupby("fator")["placebo_ganho"].mean().abs().max()), 4), 0.0, 0.01, source="acaso")
# IC 95% pareado por reamostragem de linhas inteiras (10 primeiros sorteios, 300 reamostragens), como nos notebooks 24 e 27
rng = np.random.default_rng(SEED); Lu = np.unique(LINC); ix = {l: np.where(LINC == l)[0] for l in Lu}; BT = {k_: [] for k_ in FAT}
for b in range(300):
    i = np.concatenate([ix[l] for l in rng.choice(Lu, len(Lu))])
    if y[i].sum() < 50: continue
    base_m = [auc_grupo(GU29[r]["base"][i], y[i], FERC[i]) for r in range(10)]
    for k_ in FAT: BT[k_].append(np.mean([auc_grupo(GU29[r][k_][i], y[i], FERC[i]) - base_m[r] for r in range(10)]))
RES29 = G29.groupby("fator").agg(auc_malhas_base=("auc_malhas_base", "mean"), auc_malhas=("auc_malhas", "mean"), ganho_malhas=("ganho_malhas", "mean"),
                                 sorteios_com_ganho=("ganho_malhas", lambda s: int((s > 0).sum())), ganho_pais=("ganho_pais", "mean"), placebo=("placebo_ganho", "mean")).reset_index()
RES29["ic_inf"] = [np.percentile(BT[k_], 2.5) for k_ in RES29["fator"]]; RES29["ic_sup"] = [np.percentile(BT[k_], 97.5) for k_ in RES29["fator"]]
print("bootstrap ok", round(time.time() - t0), "s"); sys.stdout.flush()

# 4. Malha inteira fora do ajuste (simula a ferrovia nova): base e base + fator, AUC na malha deixada de fora, média ponderada pelos km atingidos
LIN_M = []
for f in range(FERC.max() + 1):
    te = FERC == f; tr = ~te
    if y[te].sum() < 5: continue
    sb = lr(XB0[tr], y[tr], XB0[te]); reg = dict(malha=C["ferrovia"].values[te][0], atingidas=int(y[te].sum()), base=auc(y[te], sb))
    for k_ in FAT:
        Xk = np.column_stack([XB0, XF[k_]]); reg[k_] = auc(y[te], lr(Xk[tr], y[tr], Xk[te]))
    LIN_M.append(reg)
MF29 = pd.DataFrame(LIN_M); MF29.to_csv(f"{TAB}/29_malha_fora.csv", index=False)
wm = MF29["atingidas"]
RES29["malha_fora_base"] = float(np.average(MF29["base"], weights=wm)); RES29["malha_fora_com_fator"] = [float(np.average(MF29[k_], weights=wm)) for k_ in RES29["fator"]]
RES29["malhas_em_que_ajuda"] = [int((MF29[k_] > MF29["base"]).sum()) for k_ in RES29["fator"]]
# 4b. Multiplicidade (pedido da revisão externa 3): valor-p unilateral do ganho pela reamostragem de linhas e correção de Holm (1979) nos 12 fatores
RES29["p_unilateral"] = [(1 + int((np.array(BT[k_]) <= 0).sum())) / (1 + len(BT[k_])) for k_ in RES29["fator"]]
ordem_ = np.argsort(RES29["p_unilateral"].values); m_ = len(ordem_); adj_ = np.empty(m_); acc_ = 0.0
for rank_, j_ in enumerate(ordem_): acc_ = max(acc_, min(1.0, (m_ - rank_) * RES29["p_unilateral"].values[j_])); adj_[j_] = acc_
RES29["p_holm"] = adj_
# 5. Regra de decisão, aplicada como foi escrita
EST_ = EST.set_index("fator"); RR_ = RRF.groupby("fator")["ic_exclui_1"].any()
RES29["a_estimavel"] = [bool(EST_.loc[k_, "estimavel"]) for k_ in RES29["fator"]]; RES29["b_rr_ic_exclui_1"] = [bool(RR_[k_]) for k_ in RES29["fator"]]
RES29["c_ganho_ic_acima_de_zero"] = RES29["ic_inf"] > 0; RES29["entra_no_EAR_v3"] = RES29["a_estimavel"] & RES29["b_rr_ic_exclui_1"] & RES29["c_ganho_ic_acima_de_zero"]
RES29["entra_com_Holm"] = RES29["a_estimavel"] & RES29["b_rr_ic_exclui_1"] & (RES29["p_holm"] < 0.05)
RES29 = RES29.merge(EST[["fator", "malhas_com_2_classes", "celulas_fora_da_classe_dominante_da_malha"]], on="fator").sort_values("ganho_malhas", ascending=False)
display(RES29.round(4)); RES29.to_csv(f"{TAB}/29_decisao_fatores.csv", index=False)
print("fatores aprovados:", RES29.loc[RES29["entra_no_EAR_v3"], "fator"].tolist(), round(time.time() - t0), "s")
log_ia("29.3 ganho e decisão", "Claude", "FOCUS: base = faixas aferidas com pesos aprendidos; base + fator; 30 sorteios; IC por linhas; malha inteira fora; regra de decisão declarada antes",
       "29_ganho_por_sorteio.csv, 29_malha_fora.csv, 29_decisao_fatores.csv", "placebo e referência")

# 6. Conciliação com o artigo (pedido da revisão externa 3): o artigo diz que os atributos declarados não acrescentam nada mensurável dentro das malhas
#    (logística com critérios contínuos e atributos contra só os critérios contínuos, sem efeito fixo de malha: +0,009). Teste que decide, declarado antes:
#    se, com efeito fixo de malha e critérios contínuos (distância sem teto), o tráfego ainda acrescentar com IC acima de zero, a frase do artigo precisa mudar;
#    se o ganho cair para perto de zero, a distância contínua já carrega a informação do tráfego e a frase fica, com ressalva.
FE_ = oh(FERC); XTRAF = XF["Tráfego (trens por dia vinculados)"]
MOD6 = {"contínuos": XE, "contínuos + malha": np.column_stack([XE, FE_]), "contínuos + malha + tráfego": np.column_stack([XE, FE_, XTRAF]),
        "contínuos + atributos (artigo)": X0, "contínuos + atributos + malha": np.column_stack([X0, FE_])}
L6, GU6 = [], {}
for r in range(R):
    fo = folds_por(SEGC, SEED + r); S6 = {k_: np.zeros(N) for k_ in MOD6}
    for k in range(KF):
        te = fo == k
        for k_, X_ in MOD6.items(): S6[k_][te] = lr(X_[~te], y[~te], X_[te])
    for k_ in MOD6: L6.append(dict(sorteio=r + 1, modelo=k_, auc_malhas=auc_grupo(S6[k_], y, FERC), auc_linhas=auc_grupo(S6[k_], y, LINC)))
    if r < 10: GU6[r] = S6
T6 = pd.DataFrame(L6); P6 = T6.pivot_table(index="sorteio", columns="modelo", values="auc_malhas")
PAR6 = [("contínuos + malha + tráfego", "contínuos + malha"), ("contínuos + atributos (artigo)", "contínuos"), ("contínuos + atributos + malha", "contínuos + malha")]
rng6 = np.random.default_rng(SEED); B6 = {pq: [] for pq in PAR6}
for b in range(300):
    i = np.concatenate([ix[l] for l in rng6.choice(Lu, len(Lu))])
    if y[i].sum() < 50: continue
    for a_, c_ in PAR6: B6[(a_, c_)].append(np.mean([auc_grupo(GU6[r][a_][i], y[i], FERC[i]) - auc_grupo(GU6[r][c_][i], y[i], FERC[i]) for r in range(10)]))
CONC = pd.DataFrame([dict(comparacao=f"{a_} − {c_}", ganho_malhas=float((P6[a_] - P6[c_]).mean()), ic_inf=float(np.percentile(B6[(a_, c_)], 2.5)), ic_sup=float(np.percentile(B6[(a_, c_)], 97.5)),
                          sorteios_com_ganho=int(((P6[a_] - P6[c_]) > 0).sum())) for a_, c_ in PAR6])
display(T6.groupby("modelo")[["auc_malhas", "auc_linhas"]].mean().round(3)); display(CONC.round(3)); CONC.to_csv(f"{TAB}/29_conciliacao_artigo.csv", index=False)
gate("reprodução do artigo: contínuos + atributos − contínuos, dentro das malhas", round(float(CONC.loc[1, "ganho_malhas"]), 3), 0.009, tol=0.003, source="24_diferencas.csv (+0,009)")
log_ia("29.4 conciliação", "Claude", "FOCUS: critérios contínuos com e sem efeito fixo de malha e tráfego; mesmo protocolo; IC por linhas", "29_conciliacao_artigo.csv", "reproduz o +0,009 do artigo")

exportar_registros("29")

In [ ]:
# Figura 29: risco relativo de cada classe de fator, nacional bruto e dentro das malhas ajustado pelas faixas aferidas. Estilo da dissertação.
import os, matplotlib, matplotlib.font_manager as fm
for f_ in ("lmsans10-regular.otf", "lmsans10-bold.otf", "lmsans10-oblique.otf"):
    for d_ in (f"{BASE}/data/fonts", "/usr/share/texmf/fonts/opentype/public/lm"):
        if os.path.exists(f"{d_}/{f_}"): fm.fontManager.addfont(f"{d_}/{f_}"); break
FONTE = "Latin Modern Sans" if any(f.name == "Latin Modern Sans" for f in fm.fontManager.ttflist) else "DejaVu Sans"
matplotlib.rcParams.update({"font.family": FONTE, "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#7A8089", "axes.linewidth": 0.6})
import matplotlib.pyplot as plt
INK, MUTE, FAINT, TEAL, CLAY = "#16191C", "#7A8089", "#C9CCD1", "#1E6E6E", "#BD4B2C"
RRF_ = pd.read_csv(f"{TAB}/29_risco_relativo_fatores.csv"); DEC_ = pd.read_csv(f"{TAB}/29_decisao_fatores.csv").set_index("fator")
ORDEM = ["Tráfego (trens por dia vinculados)", "Carga por eixo (t)", "Velocidade máxima carregado (km/h)", "Rampa máxima (%)", "Dormente", "Trilho", "Fixação",
         "Taxa de dormentação (por km)", "Sinalização", "Produto perigoso", "Número de linhas", "Bitola"]
linhas = []
for f_ in ORDEM:
    g = RRF_[(RRF_["fator"] == f_) & (RRF_["classe"] != RRF_["referencia"]) & (RRF_["classe"] != "sem dado") & RRF_["rr_ajustado"].notna()]
    linhas.append(("titulo", f_, None))
    for _, r_ in g.iterrows(): linhas.append(("classe", f_, r_))
fig, ax = plt.subplots(figsize=(8.4, 9.6)); yv = 0; ticks, labs = [], []
for tipo, f_, r_ in linhas:
    if tipo == "titulo":
        ok_ = bool(DEC_.loc[f_, "entra_no_EAR_v3"]); ref_ = RRF_.loc[RRF_["fator"] == f_, "referencia"].iloc[0]
        ax.text(0.052, yv, f"{f_}  ·  ref.: {ref_}" + ("  ·  aprovado" if ok_ else ""), fontsize=8.6, fontweight="bold", color=TEAL if ok_ else INK, va="center", ha="left", transform=ax.get_yaxis_transform())
        yv -= 1; continue
    ax.plot([max(r_["ic_inf"], 0.06), min(r_["ic_sup"], 9.5)], [yv, yv], color=TEAL, lw=1.4, solid_capstyle="butt", zorder=3)
    ax.plot(r_["rr_ajustado"], yv, "o", ms=5.2, color=TEAL, mec="white", mew=0.8, zorder=4)
    ax.plot(max(r_["rr_bruto_nacional"], 0.06), yv, "D", ms=4.6, mfc="white", mec=CLAY, mew=1.1, zorder=5)
    ticks.append(yv); labs.append(r_["classe"]); yv -= 1
ax.axvline(1.0, color=MUTE, lw=0.8, ls=(0, (4, 3)), zorder=1); ax.set_xscale("log"); ax.set_xlim(0.06, 10); ax.set_ylim(yv + 0.5, 0.8)
ax.set_xticks([0.1, 0.25, 0.5, 1, 2, 4, 8]); ax.set_xticklabels(["0,1", "0,25", "0,5", "1", "2", "4", "8"])
ax.set_yticks(ticks); ax.set_yticklabels(labs, fontsize=8.2); ax.tick_params(axis="y", length=0); ax.spines["left"].set_visible(False)
ax.set_xlabel("Risco relativo de descarrilamento por via (escala logarítmica)", fontsize=9)
ax.plot([], [], "D", mfc="white", mec=CLAY, label="nacional, bruto (compara ferrovias)"); ax.plot([], [], "o", color=TEAL, label="dentro das malhas, ajustado por faixa de pátio e raio (IC 95%)")
ax.legend(loc="lower center", bbox_to_anchor=(0.45, 1.0), ncol=2, fontsize=8, frameon=False)
fig.text(0.01, 0.005, "Fonte: o autor, a partir do RAAF e das Declarações de Rede 2025 (ANTT), 2011 a 2025; notebook 29. Aprovado: atende aos três critérios declarados antes.", fontsize=7.2, color=MUTE)
fig.tight_layout(rect=(0, 0.015, 1, 0.985)); fig.savefig(f"{FIG}/29_fatores_dentro_das_malhas_PT.png", dpi=220); plt.close(fig)
print("figura 29 ok")


---
## Referências

Mantel, N., & Haenszel, W. (1959). Statistical aspects of the analysis of data from retrospective studies of disease. *Journal of the National Cancer Institute, 22*(4), 719–748.

Roberts, D. R., et al. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography, 40*(8), 913–929. https://doi.org/10.1111/ecog.02881

**Declaração de uso de IA (regra da disciplina, três partes).** Ferramentas: Claude (Anthropic), usado pelo autor como assistente de programação a partir das especificações FOCUS registradas neste notebook, e Gemini do Colab quando indicado. Parte do trabalho: geração do código de cada passo e revisão do texto das células. Responsável pelo conteúdo: o autor, que definiu o problema, as fontes, as especificações, os gates e a verificação, e executou o notebook do zero antes de reportar qualquer número.